# **El Gran Director**

In [4]:
!mkdir -p grandirector/constants

In [5]:
%%writefile grandirector/GranDirector.java
package grandirector;

import java.io.File;
import java.io.FileNotFoundException;
import java.nio.file.Path;
import java.util.ArrayList;
import java.util.List;
import java.util.Scanner;

public class GranDirector
{
    public static void main(String[] args) throws FileNotFoundException
    {
        if(args[0] == null )
            throw new FileNotFoundException();
        String fileName = args[0];

        List<Team> teams = readTeamsFile(fileName);

        Fixture fixture = new Fixture(teams);
        System.out.println(fixture);

        Tournament apertura98 = new Tournament("Apertura 92", fixture);

        long startTime = System.currentTimeMillis();
        apertura98.startConcurrent();
        apertura98.showResults();

        System.out.println("Tiempo total: " + (System.currentTimeMillis() - startTime) / 1000.0 + " segundos");

        apertura98.reset();
        startTime = System.currentTimeMillis();
        apertura98.startSecuential();
        apertura98.showResults();
        System.out.format("Tiempo total: %.2f segundos", (System.currentTimeMillis() - startTime) / 1000.0);

    }

    private static List<Team> readTeamsFile(String fileName)
    {
        List<Team> teams = new ArrayList<>();
        File file = new File(fileName);
        try
        {
            Scanner scanner = new Scanner(file);
            while (scanner.hasNextLine())
            {
                String data = scanner.nextLine();
                String[] splittedData = data.split(",");
                teams.add(new Team(splittedData[0], Integer.parseInt(splittedData[1], 10)));
            }
            scanner.close();
        } catch (FileNotFoundException e)
        {
            System.out.println("No se encontro el achivo de equipos");
            throw new RuntimeException(e);
        }
        return teams;
    }
}

Writing grandirector/GranDirector.java


In [6]:
%%writefile grandirector/Fixture.java
package grandirector;

import java.util.ArrayList;
import java.util.Collections;
import java.util.List;

public class Fixture
{

    private final List<Team> teams;
    private List<GameWeek> weeks;

    public Fixture(List<Team> teams)
    {
        this.teams = teams;
        generateMatches();
    }

    private void generateMatches()
    {
        List<Team> teamsAux = new ArrayList<>(teams);
        Collections.shuffle(teamsAux);
        int teamsSize = teamsAux.size();
        int numRounds = teamsSize - 1;
        int teamsSizeHalf = teamsSize / 2;

        this.weeks = new ArrayList<>();

        for (int round = 0; round < numRounds; round++)
        {
            List<Match> weekMatches = new ArrayList<>();
            for (int i = 0; i < teamsSizeHalf; i++)
            {
                Team home = teamsAux.get(i);
                Team away = teamsAux.get(teamsSize - 1 - i);

                if (round % 2 == 0)
                {
                    weekMatches.add(new Match(home, away));
                } else
                {
                    weekMatches.add(new Match(away, home));
                }
            }

            weeks.add(new GameWeek(Integer.toString(round + 1), weekMatches));

            Team fixed = teamsAux.get(0);
            teamsAux.remove(0);
            teamsAux.add(1, fixed);
            Collections.rotate(teamsAux.subList(1, teamsSize), 1);
        }
    }

    @Override
    public String toString()
    {
        String text = "--\n";
        for (GameWeek week : weeks)
        {
            text = text.concat(week.toString()).concat("--\n");
        }
        return text;
    }

    public List<Team> getTeams()
    {
        return teams;
    }

    public List<GameWeek> getWeeks()
    {
        return weeks;
    }
}


Writing grandirector/Fixture.java


In [7]:
%%writefile grandirector/GameWeek.java
package grandirector;

import java.util.List;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.ExecutorService;

public class GameWeek
{

    private final String gameWeekNumber;
    private final List<Match> matches;

    public GameWeek(String gameWeekNumber, List<Match> matches)
    {
        this.gameWeekNumber = gameWeekNumber;
        this.matches = matches;
    }

    public List<Match> playConcurrent(ExecutorService executor, int poolSize)
    {
        System.out.println("Jugando fecha " + this.gameWeekNumber);
        CountDownLatch latch = new CountDownLatch(poolSize);

        for (Match match : matches)
        {
            executor.submit(() ->
            {
                match.run();
                latch.countDown();
            });
        }

        try
        {
            latch.await();
        } catch (InterruptedException e)
        {
            throw new RuntimeException(e);
        }

        return matches;
    }

    public List<Match> playSecuential()
    {
        System.out.println("Jugando fecha " + this.gameWeekNumber);
        for (Match match : matches)
        {
            match.play();
        }

        return matches;
    }

    @Override
    public String toString()
    {
        String text = "Fecha " + gameWeekNumber + ":\n";
        for (Match match : matches)
        {
            text = text.concat(match.toString() + "\n");
        }
        return text;
    }

    public List<Match> getMatches()
    {
        return matches;
    }
}


Writing grandirector/GameWeek.java


In [8]:
%%writefile grandirector/Match.java
package grandirector;

import grandirector.constants.Constants;

import java.util.Random;

public class Match implements Runnable
{
    private final Team home;
    private final Team away;
    private MatchResult matchResult;

    public Match(Team home, Team away)
    {
        this.home = home;
        this.away = away;
    }

    @Override
    public String toString()
    {
        return home.getName() + " vs " + away.getName();
    }

    public MatchResult play()
    {
        System.out.println("Iniciando " + this.toString());
        Integer powerHome = home.getPower();
        Integer powerAway = away.getPower();
        int pointsHome;
        int pointsAway;

        int goalsHome = 0;
        int goalsAway = 0;

        Random rand = new Random();
        long startTime = System.currentTimeMillis();
        long duration = rand.nextInt(Constants.MAX_MATCH_DURATION - Constants.MIN_MATCH_DURATION) + Constants.MIN_MATCH_DURATION;

        while (System.currentTimeMillis() < (startTime + duration))
        {
            pointsHome = (int) Math.round(rand.nextGaussian() * powerHome);
            pointsAway = (int) Math.round(rand.nextGaussian() * powerAway);

            if (pointsAway * Constants.PERCENTAGE_DIFFERENCE_TO_GOAL > Math.abs(pointsHome - pointsAway))
                if (pointsAway > pointsHome)
                    goalsAway++;
                else
                    goalsHome++;
            try
            {
                Thread.sleep(Constants.MS_BETWEEN_MATCH_TICKS);
            } catch (InterruptedException e)
            {
                System.out.println("Error en partido");
            }
        }
        this.matchResult = new MatchResult(goalsHome, goalsAway);
        System.out.println("Resultado " + this.toString() + ": " + this.matchResult + " Duracion: " + (System.currentTimeMillis() - startTime) + "ms");
        return this.matchResult;
    }

    @Override
    public void run()
    {
        play();
    }

    public MatchResult getResult()
    {
        return this.matchResult;
    }

    public Team getHome()
    {
        return home;
    }

    public Team getAway()
    {
        return away;
    }
}


Writing grandirector/Match.java


In [9]:
%%writefile grandirector/MatchResult.java
package grandirector;

public class MatchResult
{
    int goalsHome;
    int goalsAway;

    public MatchResult(int goalsHome, int goalsAway)
    {
        this.goalsHome = goalsHome;
        this.goalsAway = goalsAway;
    }


    public int getGoalsAway()
    {
        return goalsAway;
    }

    public int getGoalsHome()
    {
        return goalsHome;
    }

    @Override
    public String toString()
    {
        return goalsHome + " - " + goalsAway;
    }

    public ResultType getResult()
    {
        if (goalsHome > goalsAway)
            return ResultType.HOME_WON;
        if (goalsAway > goalsHome)
            return ResultType.AWAY_WON;
        return ResultType.TIE;
    }
}


Writing grandirector/MatchResult.java


In [10]:
%%writefile grandirector/ResultType.java
package grandirector;

public enum ResultType
{
    TIE,
    HOME_WON,
    AWAY_WON;
}


Writing grandirector/ResultType.java


In [11]:
%%writefile grandirector/Stats.java
package grandirector;

public class Stats implements Comparable<Stats>
{
    int points = 0;
    int played = 0;
    int won = 0;
    int tied = 0;
    int lost = 0;
    int goalsInFavor = 0;
    int goalsAgainst = 0;
    int goalsDiff = 0;

    public void addVictory(MatchResult result)
    {
        this.played++;
        this.won++;
        this.points += 3;
        this.goalsInFavor += Math.max(result.getGoalsAway(), result.getGoalsHome());
        this.goalsAgainst += Math.min(result.getGoalsAway(), result.getGoalsHome());
        this.goalsDiff += Math.abs(result.getGoalsAway() - result.getGoalsHome());
    }

    public void addLost(MatchResult result)
    {
        this.played++;
        this.lost++;
        this.goalsInFavor += Math.min(result.getGoalsAway(), result.getGoalsHome());
        this.goalsAgainst += Math.max(result.getGoalsAway(), result.getGoalsHome());
        this.goalsDiff -= Math.abs(result.getGoalsAway() - result.getGoalsHome());
    }

    public void addTie(MatchResult result)
    {
        this.played++;
        this.tied++;
        this.points += 1;
        this.goalsInFavor += result.getGoalsAway();
        this.goalsAgainst += result.getGoalsAway();
    }

    @Override
    public int compareTo(Stats otherStats)
    {
        int result = Integer.compare(this.points, otherStats.points);
        if (result != 0)
        {
            return result;
        }

        result = Integer.compare(this.goalsDiff, otherStats.goalsDiff);
        if (result != 0)
        {
            return result;
        }

        return Integer.compare(this.goalsInFavor, otherStats.goalsInFavor);
    }

    public int getPoints()
    {
        return points;
    }

    public int getGoalsDiff()
    {
        return goalsDiff;
    }

    public int getGoalsAgainst()
    {
        return goalsAgainst;
    }

    public int getGoalsInFavor()
    {
        return goalsInFavor;
    }

    public int getLost()
    {
        return lost;
    }

    public int getTied()
    {
        return tied;
    }

    public int getWon()
    {
        return won;
    }

    public int getPlayed()
    {
        return played;
    }
}


Writing grandirector/Stats.java


In [12]:
%%writefile grandirector/Team.java
package grandirector;

public class Team
{
    private final String name;
    private final Integer power;

    public Team(String name, Integer power)
    {
        this.name = name;
        this.power = power;
    }

    public String getName()
    {
        return name;
    }

    public Integer getPower()
    {
        return power;
    }
}


Writing grandirector/Team.java


In [13]:
%%writefile grandirector/Tournament.java
package grandirector;

import java.util.*;
import java.util.concurrent.ExecutorService;
import java.util.concurrent.Executors;
import java.util.stream.Collectors;

public class Tournament
{
    private final Fixture fixture;
    private Map<Team, Stats> teamsStats;

    public Tournament(String name, Fixture fixture)
    {
        this.fixture = fixture;
        this.teamsStats = generatePositionTable();
    }

    public void showResults()
    {
        String leftAlignFormat = "| %-2d | %-16s | %-3d | %-2d | %-2d | %-2d | %-2d | %-2d | %-2d | %-3d |%n";

        System.out.format("+---+-------------------+-----+----+----+----+----+----+----+-----+%n");
        System.out.format("| # | Equipo            | Pts | PJ | PG | PE | PP | GF | GC | DIF |%n");
        System.out.format("+---+-------------------+-----+----+----+----+----+----+----+-----+%n");
        List<Team> teamsInOrder = this.teamsStats.keySet().stream().sorted((o1, o2) -> this.teamsStats.get(o2).compareTo(this.teamsStats.get(o1))).collect(Collectors.toList());

        for (int i = 0; i < teamsInOrder.size(); i++)
        {
            Team team = teamsInOrder.get(i);
            Stats stats = this.teamsStats.get(team);
            System.out.format(leftAlignFormat, i + 1, team.getName(), stats.getPoints(), stats.getPlayed(), stats.getWon(), stats.getTied(), stats.getLost(), stats.getGoalsInFavor(), stats.getGoalsAgainst(), stats.getGoalsDiff());
        }
        System.out.format("+---+-------------------+-----+----+----+----+----+----+----+-----+%n");
    }

    public void startConcurrent()
    {
        List<GameWeek> weeks = this.fixture.getWeeks();

        int poolSize = weeks.get(0).getMatches().size();
        ExecutorService executor = Executors.newFixedThreadPool(poolSize);

        for (GameWeek week : weeks)
        {
            List<Match> weekMatches = week.playConcurrent(executor, poolSize);
            processWeekResults(weekMatches);
        }
        executor.shutdown();
    }

    public void startSecuential()
    {
        List<GameWeek> weeks = this.fixture.getWeeks();

        for (GameWeek week : weeks)
        {
            List<Match> weekMatches = week.playSecuential();
            processWeekResults(weekMatches);
        }
    }

    public void reset()
    {
        this.teamsStats = generatePositionTable();
    }

    private Map<Team, Stats> generatePositionTable()
    {
        Map<Team, Stats> teamStatsMap = new HashMap<>();
        for (Team team : fixture.getTeams())
        {
            teamStatsMap.put(team, new Stats());
        }
        return teamStatsMap;
    }

    private void processWeekResults(List<Match> matches)
    {
        for (Match match : matches)
        {
            processSingleResult(match);
        }
    }

    private void processSingleResult(Match match)
    {
        MatchResult matchResult = match.getResult();
        ResultType resultType = matchResult.getResult();

        Stats homeStats = this.teamsStats.get(match.getHome());
        Stats awayStats = this.teamsStats.get(match.getAway());

        if (resultType.equals(ResultType.HOME_WON))
        {
            homeStats.addVictory(matchResult);
            awayStats.addLost(matchResult);
        }

        if (resultType.equals(ResultType.AWAY_WON))
        {
            awayStats.addVictory(matchResult);
            homeStats.addLost(matchResult);
        }

        if (resultType.equals(ResultType.TIE))
        {
            awayStats.addTie(matchResult);
            homeStats.addTie(matchResult);
        }
    }
}

Writing grandirector/Tournament.java


In [14]:
%%writefile grandirector/constants/Constants.java
package grandirector.constants;

public class Constants
{

    private Constants()
    {

    }

    public static final int MIN_MATCH_DURATION = 100;
    public static final int MAX_MATCH_DURATION = 500;

    public static final int MS_BETWEEN_MATCH_TICKS = 5;

    public static final double PERCENTAGE_DIFFERENCE_TO_GOAL = 0.2;

}


Writing grandirector/constants/Constants.java


In [15]:
%%writefile grandirector/teamsData.txt
Boca Juniors,90
River Plate,88
San Lorenzo,85
Ferro,80
Huracan,78
Velez,77
Estudiantes (LP),75
Belgrano,74
Lanus,74
Talleres (C),72
Dep Español,70
San Martín (T),68
Dep Mandiyu (C),68
Rosario Central,67
Independiente,66
Racing Club,65
Gimnasia (LP),64
Platense,63
Argentinos,62
Newells,60


Writing grandirector/teamsData.txt


In [16]:
!find grandirector -name "*.java" > grandirector/sources.txt
!javac @grandirector/sources.txt

In [17]:
!java grandirector/GranDirector grandirector/teamsData.txt

--
Fecha 1:
Gimnasia (LP) vs San Lorenzo
Dep Español vs Lanus
Velez vs Newells
Rosario Central vs Talleres (C)
Estudiantes (LP) vs Dep Mandiyu (C)
Platense vs Ferro
San Martín (T) vs Argentinos
Huracan vs Racing Club
Belgrano vs Independiente
River Plate vs Boca Juniors
--
Fecha 2:
Lanus vs Dep Español
Newells vs San Lorenzo
Talleres (C) vs Gimnasia (LP)
Dep Mandiyu (C) vs Velez
Ferro vs Rosario Central
Argentinos vs Estudiantes (LP)
Racing Club vs Platense
Independiente vs San Martín (T)
Boca Juniors vs Huracan
River Plate vs Belgrano
--
Fecha 3:
San Lorenzo vs Newells
Lanus vs Talleres (C)
Dep Español vs Dep Mandiyu (C)
Gimnasia (LP) vs Ferro
Velez vs Argentinos
Rosario Central vs Racing Club
Estudiantes (LP) vs Independiente
Platense vs Boca Juniors
San Martín (T) vs River Plate
Huracan vs Belgrano
--
Fecha 4:
Talleres (C) vs Lanus
Dep Mandiyu (C) vs Newells
Ferro vs San Lorenzo
Argentinos vs Dep Español
Racing Club vs Gimnasia (LP)
Independiente vs Velez
Boca Juniors vs Rosario Cen